# Inspect a session

Loads the derived files of one processed session and draws the pipeline's debug plots (PLAN §12.2, §12.3) for ad-hoc exploration. The plots are the pipeline's own: they are imported from `backend/worker/pipeline/plots.py`, so there is one definition of each picture.

Run it with the backend's environment, from this directory, e.g. `cd backend && uv run --with jupyterlab jupyter lab ../notebooks/inspect_session.ipynb`. Nothing is written: the notebook only reads `derived/` and `raw/events.jsonl` (and `raw/browser_features.jsonl` for the brightness plot). Files are described in `docs/DATA_FORMATS.md`.

In [ ]:
# The one thing to set: the session's derived/ directory,
# <data root>/studies/<study id>/sessions/<session id>/derived  (relative paths start in notebooks/).
# The environment variable INSPECT_SESSION_DERIVED takes precedence (the test uses it).
import os

DERIVED = os.environ.get(
    "INSPECT_SESSION_DERIVED", "../data/studies/<study>/sessions/<id>/derived"
)

In [ ]:
import dataclasses
import json
import sys
import uuid
from pathlib import Path

import pandas as pd
from IPython.display import Image, display

sys.path.insert(0, str(Path("../backend").resolve()))  # worker/ and app/ live there

from app.stage_status import STAGES_FILE, read_entries
from worker.pipeline.config import PipelineConfig
from worker.pipeline.plots import (
    PlotSkipped,
    load_plot_data,
    plot_calibration,
    plot_gaze,
    plot_signals,
    plot_sync,
    plot_validation,
    png_bytes,
)
from worker.pipeline.spec import SPEC_A, SPEC_B

derived = Path(DERIVED).resolve()
if not (derived / "quality.json").is_file():
    raise FileNotFoundError(f"{derived} is not a processed session's derived/ directory")
session_dir = derived.parent  # <root>/studies/<study>/sessions/<session>
session_id = uuid.UUID(session_dir.name)
study_id = int(session_dir.parent.parent.name)
root = session_dir.parent.parent.parent.parent

quality = json.loads((derived / "quality.json").read_text(encoding="utf-8"))
known = {f.name for f in dataclasses.fields(PipelineConfig)}
config = PipelineConfig(**{
    k: tuple(v) if isinstance(v, list) else v for k, v in quality["config"].items() if k in known
})  # the thresholds this session was processed with
print(f"study {study_id}, session {session_id}")

## Quality report and stages

In [ ]:
inclusion = quality["inclusion"]
print("included:", inclusion["included"], "| reasons:", inclusion["reasons"] or "none")
print("criteria:", inclusion["criteria"])
print("calibration:", {k: quality["calibration"].get(k) for k in ("status", "failed_reason", "attempt_used")})
print("webcam:", quality["webcam"])
print("playback:", {k: v for k, v in (quality["playback"] or {}).items() if k != "per_video"})
print("sync flags:", quality["sync"]["flags"] or "none")


def stage_table(filename):
    entries = read_entries(derived, quality, filename)
    return pd.DataFrame(
        [{"stage": n, "status": e["status"], "seconds": e["seconds"], "error": e["error"]}
         for n, e in entries.items()]
    )


display(stage_table(STAGES_FILE))

## Pipeline A

Gaze x and y over time (shaded: calibration, validation and playback phases; green bars: fixations), the calibration targets against the model's predictions, the validation error per block, and the sync and signal plots. A plot whose inputs are missing says why.

In [ ]:
def show(draw, data):
    try:
        display(Image(png_bytes(draw(data))))
    except PlotSkipped as reason:
        print(f"{draw.__name__}: skipped, {reason}")


data_a = load_plot_data(root, study_id, session_id, config, SPEC_A)
for draw in (plot_gaze, plot_calibration, plot_validation, plot_sync, plot_signals):
    show(draw, data_a)

## Pipeline B

Drawn only when `gaze_b.parquet` exists: the same gaze, calibration and validation plots from B's files.

In [ ]:
if (derived / SPEC_B.gaze_file).is_file():
    data_b = load_plot_data(root, study_id, session_id, config, SPEC_B)
    display(stage_table("stages_b.json"))
    for draw in (plot_gaze, plot_calibration, plot_validation):
        show(draw, data_b)
else:
    print("Pipeline B has not run for this session (no gaze_b.parquet).")

## Free exploration

The tables are plain Parquet; `data_a.gaze` and `data_a.fixations` are Arrow tables (`.to_pandas()`).

In [ ]:
if data_a.gaze is not None:
    display(data_a.gaze.to_pandas().describe().T)